# 使用 Bedrock AgentCore Browser 的 Strands Agents

本实验演示如何将 Strands Agents 与 Amazon Bedrock AgentCore Browser 集成，创建能够动态与 Web 浏览器交互的 AI 代理。

## 概述

在本实验中，您将：

- 了解 Bedrock AgentCore Browser 的功能
- 使用 Strands Agents 测试浏览器自动化
- 以编程方式浏览网站并提取信息
- 探索浏览器自动化的常见用例
- 了解 Web 交互的最佳实践


## 前提条件

在开始本实验之前，请确保您已具备以下条件：

- 已配置 AWS 凭证（IAM 角色或环境变量）
- 已安装所需的 Python 包
- 基于 AWS 区域的 Nova Pro 模型 ID

如果您未在已承担 IAM 角色的环境中运行，请将 AWS 凭证设置为环境变量：


In [ ]:
# %env AWS_REGION=<Bedrock AgentCore Region>
# %env AWS_ACCESS_KEY_ID=<YOUR ACCESS KEY>
# %env AWS_SECRET_ACCESS_KEY=<YOUR SECRET KEY>
# %env AWS_SESSION_TOKEN=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
# %env AWS_PROFILE=<OPTIONAL - Your profile configured in ~/.aws/config>

安装 Strands Agents、Playwright 和 Bedrock AgentCore Python SDK 所需的包：


In [ ]:
#%pip install -q strands-agents 'strands-agents-tools[agent_core_browser]' bedrock-agentcore playwright rich

根据 AWS 区域设置 Nova Pro 模型 ID：

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

## 什么是 Bedrock AgentCore Browser？

Amazon Bedrock AgentCore Browser 是一款强大的工具，使 AI 代理能够在安全的托管环境中动态地与 Web 浏览器进行交互。主要功能包括：

- **Web 导航**：以编程方式浏览网站、点击元素和填写表单
- **内容提取**：从网页中提取信息并捕获屏幕截图
- **安全环境**：在隔离的安全浏览器环境中运行
- **JavaScript 执行**：执行自定义 JavaScript 以实现高级 Web 交互
- **会话管理**：跨多个操作维护浏览器会话

Browser 工具使代理能够执行需要视觉理解和交互能力的复杂 Web 任务。


### 访问浏览器会话

首先，让我们在没有任何代理环境的情况下探索 AgentCore Browser 工具。我们将使用 Playwright 库连接到远程浏览器会话。

连接后，我们可以像真实用户一样开始与页面元素进行交互。


In [ ]:
import boto3
import rich
from bedrock_agentcore.tools.browser_client import browser_session
from playwright.async_api import async_playwright
from IPython.display import display_jpeg

console = rich.get_console()
region = boto3.Session().region_name or "us-east-1"

with browser_session(region) as client:
    console.print(f"🌐 Using AgentCore Browser Session: {client.session_id}", style="cyan")
    ws_url, headers = client.generate_ws_headers()

    async with async_playwright() as playwright:
        chromium = playwright.chromium
        browser = await chromium.connect_over_cdp(endpoint_url=ws_url, headers=headers)
        console.print("[green]✅ Browser connected.[/green]")

        context = browser.contexts[0] if browser.contexts else await browser.new_context()
        page = context.pages[0] if context.pages else await context.new_page()

        console.print("[cyan]🔄 Navigating to AWS Builder Center[/cyan]")
        await page.goto("https://builder.aws.com/")
        await page.wait_for_timeout(5000)
        display_jpeg(await page.screenshot(), raw=True)

        console.print("[cyan]🔄 Clicking on the first article in the spotlight[/cyan]")
        await page.click(".swiper-wrapper div a")
        await page.wait_for_timeout(5000)
        display_jpeg(await page.screenshot(), raw=True)


### 使用浏览器自动化测试 Strands Agent

让我们演示使用 AgentCore Browser 进行 Web 自动化的 Strands Agent。我们将导航到 Amazon，搜索咖啡机，并从第一个结果中提取详细的产品信息。


In [ ]:
import boto3
import rich
from strands import Agent
from strands.models import BedrockModel
from strands_tools.browser import AgentCoreBrowser


console = rich.get_console()
region = boto3.Session().region_name or "us-east-1"

agentcore_browser = AgentCoreBrowser(region=region)

agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""你是一个通过代码执行来验证所有答案的有用 AI 助手。
                  請以中文回答用戶。""",
    tools=[agentcore_browser.browser],
)

response = await agent.invoke_async("访问 https://builder.aws.com/learn/topics/amazon-bedrock-agentcore，页面加载完成后找到第一个路径以 '/content' 开头的链接并点击它，页面加载完成后，获取页面 HTML 并总结该文章。")

console.print(response.message["content"][0]["text"] if "text" in response.message["content"][0] else "")


让我们检查代理循环的详细执行流程，以了解代理如何处理请求并生成响应：


In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

### 清理资源

In [ ]:
client = boto3.client("bedrock-agentcore")

args = {"browserIdentifier": "aws.browser.v1", "status": "READY"}
while True:
    response = client.list_browser_sessions(**args)
    for session in response["items"]:
        client.stop_browser_session(
            browserIdentifier=args["browserIdentifier"], sessionId=session["sessionId"]
        )

    if "NextToken" in response:
        args["NextToken"] = response["NextToken"]
    else:
        break

## Bedrock AgentCore Browser 的常见用例

除了 Web 搜索之外，Bedrock AgentCore Browser 还支持多种自动化场景：

### Web 应用程序测试

- 在安全环境中测试 Web 应用程序
- 验证用户界面和功能
- 执行自动化质量保证

### 数据收集与监控

- 从网站中提取信息
- 监控网站变更和更新
- 捕获屏幕截图并记录浏览器会话

### 业务流程自动化

- 自动化表单提交和数据录入
- 执行电子商务交易
- 访问在线资源和服务

### AI 驱动的 Web 交互

- 构建能够浏览 Web 的 AI 代理
- 智能地与 Web 界面交互
- 执行基于 Web 的任务和工作流

**其他资源：**
[Bedrock AgentCore Browser Use Cases Examples](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-use-cases.html#browser-use-cases-examples)
